In [2]:
import os
import pandas as pd
from PIL import Image
from collections import Counter

# =========================
# 1. SET YOUR PATHS HERE
# =========================
DATASET_ROOT = r"/kaggle/input/datasets/mohlamin/resized-eyepacs-diabetic-retinopathy-dataset"   # change this
IMAGES_DIR = os.path.join(DATASET_ROOT, "Images")
LABELS_FILE = os.path.join(DATASET_ROOT, "all_labels.csv")   # or original_train_labels.csv



In [3]:

# =========================
# 2. LOAD LABEL CSV
# =========================
df = pd.read_csv(LABELS_FILE)

print("\n===== BASIC CSV INFO =====")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())

print("\n===== NULL VALUES =====")
print(df.isnull().sum())



===== BASIC CSV INFO =====
Shape: (88700, 3)
Columns: ['image', 'level', 'Usage']

First 5 rows:
      image  level Usage
0   10_left      0   NaN
1  10_right      0   NaN
2   13_left      0   NaN
3  13_right      0   NaN
4   15_left      1   NaN

===== NULL VALUES =====
image        0
level        0
Usage    35126
dtype: int64


In [4]:

# =========================
# 3. IDENTIFY IMAGE ID COLUMN AND LABEL COLUMN
# =========================
possible_image_cols = [c for c in df.columns if "image" in c.lower() or "id" in c.lower()]
possible_label_cols = [c for c in df.columns if "label" in c.lower() or "diagnosis" in c.lower()]

print("\nPossible image/id columns:", possible_image_cols)
print("Possible label columns:", possible_label_cols)

# Manually set these after checking output if needed
IMAGE_COL = possible_image_cols[0] if possible_image_cols else df.columns[0]
LABEL_COL = possible_label_cols[0] if possible_label_cols else df.columns[1]

print("\nUsing IMAGE_COL =", IMAGE_COL)
print("Using LABEL_COL =", LABEL_COL)


Possible image/id columns: ['image']
Possible label columns: []

Using IMAGE_COL = image
Using LABEL_COL = level


In [5]:


# =========================
# 4. LABEL DISTRIBUTION
# =========================
print("\n===== LABEL DISTRIBUTION =====")
label_counts = df[LABEL_COL].value_counts().sort_index()
print(label_counts)

print("\nPercentage distribution:")
print((label_counts / len(df) * 100).round(2))


===== LABEL DISTRIBUTION =====
level
0    65342
1     6205
2    13152
3     2087
4     1914
Name: count, dtype: int64

Percentage distribution:
level
0    73.67
1     7.00
2    14.83
3     2.35
4     2.16
Name: count, dtype: float64


In [6]:

# =========================
# 5. CHECK IMAGE FILE MATCHING
# =========================
print("\n===== IMAGE FILE CHECK =====")

def find_image_file(image_id):
    for ext in [".png", ".jpg", ".jpeg"]:
        path = os.path.join(IMAGES_DIR, str(image_id) + ext)
        if os.path.exists(path):
            return path
    return None

df["image_path"] = df[IMAGE_COL].apply(find_image_file)
missing_df = df[df["image_path"].isnull()]

print("Total records in CSV:", len(df))
print("Matched image files:", df["image_path"].notnull().sum())
print("Missing image files:", len(missing_df))

if len(missing_df) > 0:
    print("\nSample missing entries:")
    print(missing_df[[IMAGE_COL, LABEL_COL]].head())



===== IMAGE FILE CHECK =====
Total records in CSV: 88700
Matched image files: 88700
Missing image files: 0


In [7]:

# =========================
# 6. LEFT / RIGHT EYE INFO
# =========================
print("\n===== LEFT / RIGHT EYE CHECK =====")
left_count = df[IMAGE_COL].astype(str).str.contains("_left").sum()
right_count = df[IMAGE_COL].astype(str).str.contains("_right").sum()

print("Images with '_left':", left_count)
print("Images with '_right':", right_count)

# Optional patient id extraction
def extract_patient_id(name):
    name = str(name)
    if "_left" in name:
        return name.replace("_left", "")
    elif "_right" in name:
        return name.replace("_right", "")
    return name

df["patient_id_guess"] = df[IMAGE_COL].apply(extract_patient_id)

print("Unique patient_id_guess count:", df["patient_id_guess"].nunique())
print("Total image count:", len(df))



===== LEFT / RIGHT EYE CHECK =====
Images with '_left': 44350
Images with '_right': 44350
Unique patient_id_guess count: 44351
Total image count: 88700


In [8]:


# =========================
# 7. SAMPLE IMAGE PROPERTIES
# =========================
print("\n===== SAMPLE IMAGE PROPERTIES =====")

sample_paths = df["image_path"].dropna().sample(min(20, df["image_path"].notnull().sum()), random_state=42).tolist()

widths, heights, modes, formats = [], [], [], []
corrupted = []

for path in sample_paths:
    try:
        with Image.open(path) as img:
            widths.append(img.width)
            heights.append(img.height)
            modes.append(img.mode)
            formats.append(img.format)
    except Exception as e:
        corrupted.append((path, str(e)))

if widths:
    print("Sampled images:", len(sample_paths))
    print("Width range:", min(widths), "to", max(widths))
    print("Height range:", min(heights), "to", max(heights))
    print("Image modes:", Counter(modes))
    print("Formats:", Counter(formats))

if corrupted:
    print("\nCorrupted images found:")
    for item in corrupted[:5]:
        print(item)



===== SAMPLE IMAGE PROPERTIES =====
Sampled images: 20
Width range: 512 to 512
Height range: 512 to 512
Image modes: Counter({'RGB': 20})
Formats: Counter({'PNG': 20})


In [9]:
from PIL import Image

bad_indices = []

for idx, row in df.iterrows():
    path = row["image_path"]
    try:
        with Image.open(path) as img:
            img.verify()  # check that image is not truncated
    except Exception:
        bad_indices.append(idx)

print("Corrupted images found:", len(bad_indices))

if bad_indices:
    df_clean = df.drop(index=bad_indices).reset_index(drop=True)
else:
    df_clean = df.copy()

df = df_clean
print("Clean df shape:", df.shape)

Corrupted images found: 1
Clean df shape: (88699, 5)


In [10]:
WORK_DIR = "/kaggle/working"
summary_file = os.path.join(WORK_DIR, "eyepacs_inspection_summary.csv")
df.to_csv(summary_file, index=False)

In [11]:


# =========================
# 10. FINAL USEFUL SUMMARY
# =========================
print("\n===== FINAL SUMMARY =====")
print(f"Total rows in labels file: {len(df)}")
print(f"Usable matched images: {df['image_path'].notnull().sum()}")
print(f"Missing images: {len(missing_df)}")
print(f"Unique guessed patients: {df['patient_id_guess'].nunique()}")
print("Class counts:")
print(label_counts.to_dict())


===== FINAL SUMMARY =====
Total rows in labels file: 88699
Usable matched images: 88699
Missing images: 0
Unique guessed patients: 44351
Class counts:
{0: 65342, 1: 6205, 2: 13152, 3: 2087, 4: 1914}


In [12]:
# =========================
# 11. TRAIN / VAL / TEST SPLIT
# =========================
import numpy as np
import pandas as pd


# Change this only if your label column has a different name
LABEL_COL = "level"

# Check required columns exist
required_cols = ["patient_id_guess", LABEL_COL, "image_path"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

# Get unique patients
patients = df["patient_id_guess"].unique()
print("Total unique patients:", len(patients))

# Shuffle for reproducibility
rng = np.random.RandomState(42)
rng.shuffle(patients)

# Split ratios
train_ratio = 0.70
val_ratio = 0.15
test_ratio = 0.15

n_total = len(patients)
n_train = int(train_ratio * n_total)
n_val = int(val_ratio * n_total)
n_test = n_total - n_train - n_val

train_patients = patients[:n_train]
val_patients = patients[n_train:n_train + n_val]
test_patients = patients[n_train + n_val:]

print("\nPatient split counts:")
print("Train patients:", len(train_patients))
print("Val patients:", len(val_patients))
print("Test patients:", len(test_patients))

# Assign split column
df["split"] = "train"
df.loc[df["patient_id_guess"].isin(val_patients), "split"] = "val"
df.loc[df["patient_id_guess"].isin(test_patients), "split"] = "test"

# Basic split summary
print("\nImage counts per split:")
print(df["split"].value_counts())

print("\nUnique patients per split:")
print(df.groupby("split")["patient_id_guess"].nunique())

print("\nClass distribution per split:")
split_class_dist = df.groupby("split")[LABEL_COL].value_counts().unstack().fillna(0).astype(int)
print(split_class_dist)

print("\nClass percentage distribution per split:")
split_class_pct = df.groupby("split")[LABEL_COL].value_counts(normalize=True).unstack().fillna(0) * 100
print(split_class_pct.round(2))

# Create separate dataframes for later use
train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "val"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

print("\nFinal dataframe sizes:")
print("train_df:", train_df.shape)
print("val_df:", val_df.shape)
print("test_df:", test_df.shape)

Total unique patients: 44351

Patient split counts:
Train patients: 31045
Val patients: 6652
Test patients: 6654

Image counts per split:
split
train    62087
test     13308
val      13304
Name: count, dtype: int64

Unique patients per split:
split
test      6654
train    31045
val       6652
Name: patient_id_guess, dtype: int64

Class distribution per split:
level      0     1     2     3     4
split                               
test    9858   924  1908   313   305
train  45681  4363  9264  1470  1309
val     9802   918  1980   304   300

Class percentage distribution per split:
level      0     1      2     3     4
split                                
test   74.08  6.94  14.34  2.35  2.29
train  73.58  7.03  14.92  2.37  2.11
val    73.68  6.90  14.88  2.29  2.25

Final dataframe sizes:
train_df: (62087, 6)
val_df: (13304, 6)
test_df: (13308, 6)


In [13]:
# =========================
# 12. CLASS WEIGHTS
# =========================

import torch

# Count labels only from training set
train_label_counts = train_df[LABEL_COL].value_counts().sort_index()
print("Training label counts:")
print(train_label_counts)

num_classes = 5
total_train_samples = len(train_df)

# Ensure all 5 classes exist in index order 0,1,2,3,4
train_label_counts = train_label_counts.reindex(range(num_classes), fill_value=0)

# Inverse-frequency class weights
class_weights = total_train_samples / (num_classes * train_label_counts.values.astype(float))

print("\nRaw class weights:")
for cls_idx, weight in enumerate(class_weights):
    print(f"Class {cls_idx}: {weight:.4f}")

# Optional normalized version (sometimes easier to inspect)
normalized_class_weights = class_weights / class_weights.mean()

print("\nNormalized class weights:")
for cls_idx, weight in enumerate(normalized_class_weights):
    print(f"Class {cls_idx}: {weight:.4f}")

# PyTorch tensor to use later in CrossEntropyLoss
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

print("\nclass_weights_tensor:")
print(class_weights_tensor)

Training label counts:
level
0    45681
1     4363
2     9264
3     1470
4     1309
Name: count, dtype: int64

Raw class weights:
Class 0: 0.2718
Class 1: 2.8461
Class 2: 1.3404
Class 3: 8.4472
Class 4: 9.4862

Normalized class weights:
Class 0: 0.0607
Class 1: 0.6355
Class 2: 0.2993
Class 3: 1.8862
Class 4: 2.1182

class_weights_tensor:
tensor([0.2718, 2.8461, 1.3404, 8.4472, 9.4862])


In [14]:
# =========================
# 13. IMAGE TRANSFORMS
# =========================

from torchvision import transforms

# EfficientNet-B3 is commonly used with ImageNet normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# You chose EfficientNet-B3; for your DR setup we will use 512x512 input
IMAGE_SIZE = 512

train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.02
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

print("Train transforms:", train_transforms)
print("\nVal transforms:", val_transforms)
print("\nTest transforms:", test_transforms)

Train transforms: Compose(
    Resize(size=(512, 512), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    RandomRotation(degrees=[-15.0, 15.0], interpolation=nearest, expand=False, fill=0)
    ColorJitter(brightness=(0.85, 1.15), contrast=(0.85, 1.15), saturation=(0.9, 1.1), hue=(-0.02, 0.02))
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

Val transforms: Compose(
    Resize(size=(512, 512), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

Test transforms: Compose(
    Resize(size=(512, 512), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


In [15]:
# =========================
# 14. CUSTOM PYTORCH DATASET
# =========================

import os
from PIL import Image
from torch.utils.data import Dataset

class EyePACSDataset(Dataset):
    def __init__(self, dataframe, image_col, label_col, transform=None):
        self.dataframe = dataframe.reset_index(drop=True).copy()
        self.image_col = image_col
        self.label_col = label_col
        self.transform = transform

        # Basic safety check
        if self.image_col not in self.dataframe.columns:
            raise ValueError(f"Column '{self.image_col}' not found in dataframe.")
        if self.label_col not in self.dataframe.columns:
            raise ValueError(f"Column '{self.label_col}' not found in dataframe.")

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        image_path = row[self.image_col]
        label = int(row[self.label_col])

        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image not found: {image_path}")

        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, label

In [16]:
# Set this correctly based on your dataframe column name
LABEL_COL = "level"  
train_dataset = EyePACSDataset(
    dataframe=train_df,
    image_col="image_path",
    label_col=LABEL_COL,
    transform=train_transforms
)

val_dataset = EyePACSDataset(
    dataframe=val_df,
    image_col="image_path",
    label_col=LABEL_COL,
    transform=val_transforms
)

test_dataset = EyePACSDataset(
    dataframe=test_df,
    image_col="image_path",
    label_col=LABEL_COL,
    transform=test_transforms
)

print("Train dataset size:", len(train_dataset))
print("Val dataset size:", len(val_dataset))
print("Test dataset size:", len(test_dataset))

Train dataset size: 62087
Val dataset size: 13304
Test dataset size: 13308


In [17]:
# =========================
# 15. DATALOADERS
# =========================

from torch.utils.data import DataLoader

# You can tune these later based on GPU memory
BATCH_SIZE = 16
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Number of training batches:", len(train_loader))
print("Number of validation batches:", len(val_loader))
print("Number of test batches:", len(test_loader))

Number of training batches: 3881
Number of validation batches: 832
Number of test batches: 832


In [18]:
# =========================
# 16. EFFICIENTNET-B3 MODEL
# =========================

import torch
import torch.nn as nn
from torchvision import models
from torchvision.models import EfficientNet_B3_Weights

NUM_CLASSES = 5

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load pretrained EfficientNet-B3
weights = EfficientNet_B3_Weights.DEFAULT
model = models.efficientnet_b3(weights=weights)

# Replace the final classifier for 5-class DR classification
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)

# Move model to device
model = model.to(device)

print(model)
print("\nFinal classifier:", model.classifier)

Using device: cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 178MB/s] 


EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 40, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(40, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(40, 40, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=40, bias=False)
            (1): BatchNorm2d(40, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(40, 10, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(10, 40, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActiv

In [19]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters: 10,703,917
Trainable parameters: 10,703,917


In [20]:
# =========================
# 17. LOSS, OPTIMIZER, SCHEDULER
# =========================

import torch.nn as nn
import torch.optim as optim

# Make sure class weights are on the same device as the model
class_weights_tensor = class_weights_tensor.to(device)

# Loss function for 5-class classification
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# Optimizer: AdamW
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Simple learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=5,   # reduce LR every 5 epochs
    gamma=0.5      # halve the LR
)

print("Criterion:", criterion)
print("Optimizer:", optimizer)
print("Scheduler:", scheduler)
print("Initial LR:", optimizer.param_groups[0]["lr"])

Criterion: CrossEntropyLoss()
Optimizer: AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.0001
    lr: 0.0001
    maximize: False
    weight_decay: 0.0001
)
Scheduler: <torch.optim.lr_scheduler.StepLR object at 0x7af50a2f5d30>
Initial LR: 0.0001


In [21]:
# =========================
# 18. TRAINING + VALIDATION LOOP
# =========================

import torch
from tqdm import tqdm

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    running_correct = 0
    total_samples = 0

    for images, labels in tqdm(dataloader, desc="Training", leave=False):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        running_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    epoch_loss = running_loss / total_samples
    epoch_acc = running_correct / total_samples

    return epoch_loss, epoch_acc


def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    running_correct = 0
    total_samples = 0

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Validation", leave=False):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            preds = torch.argmax(outputs, dim=1)
            running_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    epoch_loss = running_loss / total_samples
    epoch_acc = running_correct / total_samples

    return epoch_loss, epoch_acc, all_labels, all_preds

In [22]:
# =========================
# RUN TRAINING
# =========================

NUM_EPOCHS = 10

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_val_loss = float("inf")
best_model_path = "/kaggle/working/best_efficientnet_b3_dr_.pth"

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}]")
    print("-" * 40)

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    val_loss, val_acc, val_labels, val_preds = validate_one_epoch(
        model, val_loader, criterion, device
    )

    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
    print(f"Current LR: {optimizer.param_groups[0]['lr']:.6f}")

    # Save best model based on validation loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        print(f"Best model saved to: {best_model_path}")


Epoch [1/10]
----------------------------------------


KeyboardInterrupt: 

In [ ]:
print("\nTraining complete.")
print("Best validation loss:", best_val_loss)

print("\nHistory keys:")
print(history.keys())

In [ ]:
from torchvision import models
import torch.nn as nn
import torch

NUM_CLASSES = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.efficientnet_b3(weights=None)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)
model.load_state_dict(torch.load("/kaggle/working/best_efficientnet_b3_dr_stage1.pth",
                                 map_location=device))
model = model.to(device)

In [ ]:
# =========================
# CONTINUE TRAINING: SETUP
# =========================

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

NUM_CLASSES = 5
best_model_path = "best_efficientnet_b3_dr_stage1.pth"
stage2_path = "best_efficientnet_b3_dr_stage2.pth"

# Rebuild same architecture
model = models.efficientnet_b3(weights=None)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)
model = model.to(device)

# Load best weights from stage 1
model.load_state_dict(torch.load(best_model_path, map_location=device))
print("Loaded best checkpoint from:", best_model_path)

# Same loss (with class weights)
class_weights_tensor = class_weights_tensor.to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# LOWER learning rate for fine-tuning
FINE_TUNE_LR = 3e-5
WEIGHT_DECAY = 1e-4

optimizer = optim.AdamW(
    model.parameters(),
    lr=FINE_TUNE_LR,
    weight_decay=WEIGHT_DECAY
)

# Optional: simpler scheduler (or reuse StepLR with longer step)
scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=5,
    gamma=0.5
)

print("Fine-tune LR:", optimizer.param_groups[0]["lr"])

In [ ]:
EXTRA_EPOCHS = 10
patience = 5
best_val_loss = float("inf")
epochs_without_improve = 0

fine_tune_history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(EXTRA_EPOCHS):
    print(f"\n[Fine-tune] Epoch {epoch+1}/{EXTRA_EPOCHS}")
    print("-" * 40)

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_labels, val_preds = validate_one_epoch(model, val_loader, criterion, device)

    scheduler.step()

    fine_tune_history["train_loss"].append(train_loss)
    fine_tune_history["train_acc"].append(train_acc)
    fine_tune_history["val_loss"].append(val_loss)
    fine_tune_history["val_acc"].append(val_acc)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
    print(f"Current LR: {optimizer.param_groups[0]['lr']:.6f}")

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        epochs_without_improve = 0

        # Save stage‑2 best
        torch.save(model.state_dict(), stage2_path)
        print(f"[Fine-tune] Stage‑2 best model saved to: {stage2_path}")

        # Optionally also overwrite the generic path:
        torch.save(model.state_dict(), "best_efficientnet_b3_dr.pth")
    else:
        epochs_without_improve += 1
        print(f"[Fine-tune] No improvement for {epochs_without_improve} epoch(s).")

    if epochs_without_improve >= patience:
        print("\nEarly stopping triggered.")
        break

print("\nFine-tuning complete.")
print("Best validation loss after fine-tune:", best_val_loss)

In [ ]:
import os
import shutil
import json

os.makedirs("output_models", exist_ok=True)

# 1) ALWAYS copy the current best file (exists after any training)
if os.path.exists("best_efficientnet_b3_dr.pth"):
    shutil.copy("best_efficientnet_b3_dr.pth",
                "output_models/best_efficientnet_b3_dr_current.pth")
    print("Copied current best -> output_models/best_efficientnet_b3_dr_current.pth")

# 2) If you already created a stage2 file, copy it as well
if os.path.exists("best_efficientnet_b3_dr_stage2.pth"):
    shutil.copy("best_efficientnet_b3_dr_stage2.pth",
                "output_models/best_efficientnet_b3_dr_stage2.pth")
    print("Copied stage2 -> output_models/best_efficientnet_b3_dr_stage2.pth")



# 4) Save histories if they exist in memory
try:
    with open("output_models/history_stage1.json", "w") as f:
        json.dump(history, f)
    print("Saved history_stage1.json")
except NameError:
    print("history (stage1) not found in this session.")

try:
    with open("output_models/history_stage2.json", "w") as f:
        json.dump(fine_tune_history, f)
    print("Saved history_stage2.json")
except NameError:
    print("fine_tune_history (stage2) not found in this session.")

print("\nAll available models/metrics copied into: output_models/")

In [ ]:
# =========================
# 19. METRICS: CONFUSION MATRIX, F1, KAPPA
# =========================

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    f1_score,
    cohen_kappa_score
)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative"]

def evaluate_classification_metrics(y_true, y_pred, class_names=None, normalize_cm=False):
    if class_names is None:
        class_names = [str(i) for i in sorted(np.unique(y_true))]

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    if normalize_cm:
        cm_norm = confusion_matrix(y_true, y_pred, normalize="true")
    else:
        cm_norm = None

    # F1 scores
    f1_macro = f1_score(y_true, y_pred, average="macro")
    f1_weighted = f1_score(y_true, y_pred, average="weighted")

    # Quadratic weighted kappa
    kappa_quadratic = cohen_kappa_score(y_true, y_pred, weights="quadratic")

    print("\n===== CLASSIFICATION REPORT =====")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

    print("Macro F1:", f1_macro)
    print("Weighted F1:", f1_weighted)
    print("Quadratic weighted kappa:", kappa_quadratic)

    # Plot confusion matrix
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix (Counts)")
    plt.tight_layout()
    plt.show()

    if cm_norm is not None:
        plt.figure(figsize=(6, 5))
        sns.heatmap(
            cm_norm,
            annot=True,
            fmt=".2f",
            cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names,
            vmin=0.0,
            vmax=1.0
        )
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.title("Confusion Matrix (Normalized by True Class)")
        plt.tight_layout()
        plt.show()

    return {
        "confusion_matrix": cm,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
        "kappa_quadratic": kappa_quadratic
}

In [ ]:
metrics = evaluate_classification_metrics(
    val_labels,
    val_preds,
    class_names=CLASS_NAMES,
    normalize_cm=True
)

In [ ]:
# =========================
# 20. TEST SET EVALUATION
# =========================

# 1) Load best model weights
best_model_path = "/kaggle/working/output_models/best_efficientnet_b3_dr_current.pth"

best_model = models.efficientnet_b3(weights=None)  # same architecture
in_features = best_model.classifier[1].in_features
best_model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)
best_model = best_model.to(device)

best_model.load_state_dict(torch.load(best_model_path, map_location=device))
best_model.eval()

print("Loaded best model from:", best_model_path)

# 2) Run through test_loader and collect preds
all_test_labels = []
all_test_preds = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing", leave=False):
        images = images.to(device)
        labels = labels.to(device)

        outputs = best_model(images)
        preds = torch.argmax(outputs, dim=1)

        all_test_labels.extend(labels.cpu().numpy())
        all_test_preds.extend(preds.cpu().numpy())

# 3) Compute metrics on test set
test_metrics = evaluate_classification_metrics(
    all_test_labels,
    all_test_preds,
    class_names=CLASS_NAMES,
    normalize_cm=True
)

print("\n===== TEST SET METRICS =====")
for k, v in test_metrics.items():
    if k == "confusion_matrix":
        print(k, "shape:", v.shape)
    else:
        print(k, ":", v)

In [ ]:
# =========================
# 20. TEST SET EVALUATION
# =========================

# 1) Load best model weights
best_model_path = "/kaggle/working/output_models/best_efficientnet_b3_dr_stage2.pth"

best_model = models.efficientnet_b3(weights=None)  # same architecture
in_features = best_model.classifier[1].in_features
best_model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)
best_model = best_model.to(device)

best_model.load_state_dict(torch.load(best_model_path, map_location=device))
best_model.eval()

print("Loaded best model from:", best_model_path)

# 2) Run through test_loader and collect preds
all_test_labels = []
all_test_preds = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing", leave=False):
        images = images.to(device)
        labels = labels.to(device)

        outputs = best_model(images)
        preds = torch.argmax(outputs, dim=1)

        all_test_labels.extend(labels.cpu().numpy())
        all_test_preds.extend(preds.cpu().numpy())

# 3) Compute metrics on test set
test_metrics = evaluate_classification_metrics(
    all_test_labels,
    all_test_preds,
    class_names=CLASS_NAMES,
    normalize_cm=True
)

print("\n===== TEST SET METRICS =====")
for k, v in test_metrics.items():
    if k == "confusion_matrix":
        print(k, "shape:", v.shape)
    else:
        print(k, ":", v)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix, classification_report
import torch.optim as optim

# =========================
# 1. LOAD BEST STAGE-2 MODEL
# =========================
stage2_path = "/kaggle/working/best_efficientnet_b3_dr_stage2.pth"
state_dict = torch.load(stage2_path, map_location=device)
model.load_state_dict(state_dict)
model = model.to(device)

# =========================
# 2. USE GENTLER LOADDERS
# =========================
BATCH_SIZE = 16
NUM_WORKERS = 2

train_loader_stage3b = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader_stage3b = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader_stage3b = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# =========================
# 3. CLASS COUNTS FROM TRAIN SET
# =========================
train_labels = train_df[LABEL_COL].astype(int).values
NUM_CLASSES = 5
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)

print("Class counts:", class_counts)

# =========================
# 4. GENTLER CLASS-BALANCED FOCAL LOSS
# =========================
class ClassBalancedFocalLoss(nn.Module):
    def __init__(self, samples_per_class, num_classes=5, beta=0.999, gamma=1.0):
        super().__init__()
        self.num_classes = num_classes
        self.gamma = gamma

        samples_per_class = np.array(samples_per_class, dtype=np.float32)
        effective_num = 1.0 - np.power(beta, samples_per_class)
        weights = (1.0 - beta) / (effective_num + 1e-8)
        weights = weights / np.sum(weights) * num_classes

        self.class_weights = torch.tensor(weights, dtype=torch.float32)

    def forward(self, logits, targets):
        weights = self.class_weights.to(logits.device)
        ce_loss = F.cross_entropy(logits, targets, reduction="none", weight=weights)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

criterion_stage3b = ClassBalancedFocalLoss(
    samples_per_class=class_counts,
    num_classes=NUM_CLASSES,
    beta=0.999,
    gamma=1.0
)

print("Loss ready.")

# =========================
# 5. OPTIMIZER / SCHEDULER
# =========================
LR_STAGE3B = 1e-5
optimizer_stage3b = optim.AdamW(model.parameters(), lr=LR_STAGE3B, weight_decay=1e-4)
scheduler_stage3b = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_stage3b,
    mode="max",
    factor=0.5,
    patience=1
)

# =========================
# 6. METRIC FUNCTIONS
# =========================
CLASS_NAMES = ["No_DR", "Mild", "Moderate", "Severe", "Proliferative_DR"]

def train_one_epoch_stage3b(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    epoch_kappa = cohen_kappa_score(all_labels, all_preds, weights="quadratic")

    return epoch_loss, epoch_acc, epoch_f1_macro, epoch_kappa


@torch.no_grad()
def evaluate_stage3b(model, loader, criterion, device, print_report=False):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    epoch_f1_weighted = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
    epoch_kappa = cohen_kappa_score(all_labels, all_preds, weights="quadratic")

    cm = confusion_matrix(all_labels, all_preds, labels=list(range(len(CLASS_NAMES))))
    report = classification_report(
        all_labels,
        all_preds,
        labels=list(range(len(CLASS_NAMES))),
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0
    )

    if print_report:
        print("\nConfusion Matrix (rows=true, cols=pred):")
        print(cm)
        print("\nPer-class Classification Report:")
        print(report)

    return {
        "loss": epoch_loss,
        "acc": epoch_acc,
        "f1_macro": epoch_f1_macro,
        "f1_weighted": epoch_f1_weighted,
        "kappa": epoch_kappa,
        "cm": cm,
        "report": report,
        "y_true": all_labels,
        "y_pred": all_preds
    }

# =========================
# 7. TRAIN LOOP
# =========================
EPOCHS = 3
best_val_kappa = -1
history_stage3b = []

for epoch in range(EPOCHS):
    train_loss, train_acc, train_f1, train_kappa = train_one_epoch_stage3b(
        model, train_loader_stage3b, criterion_stage3b, optimizer_stage3b, device
    )

    val_metrics = evaluate_stage3b(
        model, val_loader_stage3b, criterion_stage3b, device, print_report=True
    )

    scheduler_stage3b.step(val_metrics["kappa"])

    history_stage3b.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "train_f1_macro": train_f1,
        "train_kappa": train_kappa,
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["acc"],
        "val_f1_macro": val_metrics["f1_macro"],
        "val_f1_weighted": val_metrics["f1_weighted"],
        "val_kappa": val_metrics["kappa"]
    })

    print(f"\nEpoch [{epoch+1}/{EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train Macro F1: {train_f1:.4f} | Train Kappa: {train_kappa:.4f}")
    print(f"Val   Loss: {val_metrics['loss']:.4f} | Val   Acc: {val_metrics['acc']:.4f} | Val   Macro F1: {val_metrics['f1_macro']:.4f} | Val   Weighted F1: {val_metrics['f1_weighted']:.4f} | Val   Kappa: {val_metrics['kappa']:.4f}")

    if val_metrics["kappa"] > best_val_kappa:
        best_val_kappa = val_metrics["kappa"]
        torch.save(model.state_dict(), "/kaggle/working/best_efficientnet_b3_stage3b_balanced.pth")
        print("Saved best model to: /kaggle/working/best_efficientnet_b3_stage3b_balanced.pth")

print("\nTraining complete.")

In [ ]:
print("===== TESTING BEST STAGE 3B BALANCED MODEL =====")

# 1) Recreate model exactly as in training (if not already in memory)
# model = EfficientNet_B3_stage2(num_classes=5).to(device)  # example, use your actual class

# 2) Load best Stage 3B balanced weights
checkpoint_path = "/kaggle/working/best_efficientnet_b3_stage3b_balanced.pth"
state_dict = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(state_dict)
model = model.to(device)

# 3) Use the Stage 3B test loader and criterion you defined above
test_loader = test_loader_stage3b
criterion = criterion_stage3b

# 4) Run evaluation – notice we now get a dict
test_metrics = evaluate_stage3b(
    model, test_loader, criterion, device, print_report=True
)

# 5) Extract and print metrics
print("\n===== TEST SET METRICS (STAGE 3B BALANCED) =====")
print("confusion_matrix shape:", test_metrics["cm"].shape)
print("loss :", test_metrics["loss"])
print("accuracy :", test_metrics["acc"])
print("f1_macro :", test_metrics["f1_macro"])
print("f1_weighted :", test_metrics["f1_weighted"])
print("kappa_quadratic :", test_metrics["kappa"])

print("\n===== CLASSIFICATION REPORT =====")
print(test_metrics["report"])

print("\n===== CONFUSION MATRIX (COUNTS) =====")
print(test_metrics["cm"])

In [ ]:
# =========================================================
# STAGE 3C: Minority-class augmentation after Stage 3B balanced
# Put this AFTER stage3b_balanced
# =========================================================

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix, classification_report

# -------------------------
# 1) CONFIG
# -------------------------
NUM_CLASSES = 5
CLASS_NAMES = ["NoDR", "Mild", "Moderate", "Severe", "ProliferativeDR"]

BATCH_SIZE = 16
NUM_WORKERS = 2
EPOCHS = 3
LR_STAGE3C = 5e-6

MINORITY_CLASSES = {1, 3, 4}      # Mild, Severe, ProliferativeDR
MINORITY_AUG_PROB = 0.7

IMAGE_SIZE = 300
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

STAGE3B_PATH = "/kaggle/working/best_efficientnet_b3_stage3b_balanced.pth"
STAGE3C_PATH = "/kaggle/working/bestefficientnet_b3stage3c_minoraug.pth"

# -------------------------
# 2) TRANSFORMS
# majority = lighter transform
# minority = stronger transform
# val/test = clean eval transform
# -------------------------
majority_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=8),
    transforms.ColorJitter(brightness=0.06, contrast=0.06, saturation=0.04, hue=0.01),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

minority_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=18),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.88, 1.0), ratio=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.08, hue=0.02),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# -------------------------
# 3) DATASET WRAPPER
# Applies stronger augmentation to minority classes only during training
# -------------------------
class EyePACSDatasetStage3C(Dataset):
    def __init__(
        self,
        df,
        imagecol="image_path",
        labelcol="level",
        train=True,
        majority_transform=None,
        minority_transform=None,
        eval_transform=None,
        minority_classes=None,
        minority_aug_prob=0.7
    ):
        self.df = df.reset_index(drop=True).copy()
        self.imagecol = imagecol
        self.labelcol = labelcol
        self.train = train
        self.majority_transform = majority_transform
        self.minority_transform = minority_transform
        self.eval_transform = eval_transform
        self.minority_classes = minority_classes if minority_classes is not None else set()
        self.minority_aug_prob = minority_aug_prob

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row[self.imagecol]
        label = int(row[self.labelcol])

        image = Image.open(img_path).convert("RGB")

        if self.train:
            if (label in self.minority_classes) and (random.random() < self.minority_aug_prob):
                image = self.minority_transform(image)
            else:
                image = self.majority_transform(image)
        else:
            image = self.eval_transform(image)

        return image, label

# -------------------------
# 4) DATASETS / LOADERS
# -------------------------
train_dataset_stage3c = EyePACSDatasetStage3C(
    df=train_df,
    imagecol="image_path",
    labelcol=LABEL_COL,
    train=True,
    majority_transform=majority_transform,
    minority_transform=minority_transform,
    eval_transform=eval_transform,
    minority_classes=MINORITY_CLASSES,
    minority_aug_prob=MINORITY_AUG_PROB
)

val_dataset_stage3c = EyePACSDatasetStage3C(
    df=val_df,
    imagecol="image_path",
    labelcol=LABEL_COL,
    train=False,
    eval_transform=eval_transform
)

test_dataset_stage3c = EyePACSDatasetStage3C(
    df=test_df,
    imagecol="image_path",
    labelcol=LABEL_COL,
    train=False,
    eval_transform=eval_transform
)

trainloader_stage3c = DataLoader(
    train_dataset_stage3c,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

valloader_stage3c = DataLoader(
    val_dataset_stage3c,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

testloader_stage3c = DataLoader(
    test_dataset_stage3c,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Stage 3C loaders ready.")
print("Train batches:", len(trainloader_stage3c))
print("Val batches:", len(valloader_stage3c))
print("Test batches:", len(testloader_stage3c))

# -------------------------
# 5) RECREATE MODEL + LOAD BEST STAGE 3B
# -------------------------
model = models.efficientnet_b3(weights=None)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)
model = model.to(device)

if not os.path.exists(STAGE3B_PATH):
    raise FileNotFoundError(f"Stage 3B checkpoint not found: {STAGE3B_PATH}")

state_dict = torch.load(STAGE3B_PATH, map_location=device)
model.load_state_dict(state_dict)
model = model.to(device)

print(f"Loaded Stage 3B checkpoint from: {STAGE3B_PATH}")

# -------------------------
# 6) CLASS-BALANCED FOCAL LOSS
# keep same philosophy as Stage 3B, but slightly gentler
# -------------------------
train_labels = train_df[LABEL_COL].astype(int).values
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)

class ClassBalancedFocalLoss(nn.Module):
    def __init__(self, samples_per_class, num_classes=5, beta=0.9995, gamma=0.75):
        super().__init__()
        self.num_classes = num_classes
        self.gamma = gamma

        samples_per_class = np.array(samples_per_class, dtype=np.float32)
        effective_num = 1.0 - np.power(beta, samples_per_class)
        weights = (1.0 - beta) / (effective_num + 1e-8)
        weights = weights / np.sum(weights) * num_classes
        self.class_weights = torch.tensor(weights, dtype=torch.float32)

    def forward(self, logits, targets):
        weights = self.class_weights.to(logits.device)
        ce_loss = F.cross_entropy(logits, targets, reduction='none', weight=weights)
        pt = torch.exp(-ce_loss)
        loss = ((1 - pt) ** self.gamma) * ce_loss
        return loss.mean()

criterion_stage3c = ClassBalancedFocalLoss(
    samples_per_class=class_counts,
    num_classes=NUM_CLASSES,
    beta=0.9995,
    gamma=0.75
)

print("Stage 3C loss ready.")

# -------------------------
# 7) OPTIMIZER / SCHEDULER
# -------------------------
optimizer_stage3c = optim.AdamW(model.parameters(), lr=LR_STAGE3C, weight_decay=1e-4)

scheduler_stage3c = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_stage3c,
    mode="max",
    factor=0.5,
    patience=1
)

# -------------------------
# 8) TRAIN / EVAL FUNCTIONS
# -------------------------
def train_one_epoch_stage3c(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    epoch_kappa = cohen_kappa_score(all_labels, all_preds, weights="quadratic")

    return epoch_loss, epoch_acc, epoch_f1_macro, epoch_kappa


@torch.no_grad()
def evaluate_stage3c(model, loader, criterion, device, print_report=False):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    epoch_f1_weighted = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
    epoch_kappa = cohen_kappa_score(all_labels, all_preds, weights="quadratic")
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(NUM_CLASSES)))
    report = classification_report(
        all_labels,
        all_preds,
        labels=list(range(NUM_CLASSES)),
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0
    )

    if print_report:
        print("Confusion Matrix (rows=true, cols=pred):")
        print(cm)
        print("\nClassification Report:\n")
        print(report)

    return {
        "loss": epoch_loss,
        "acc": epoch_acc,
        "f1_macro": epoch_f1_macro,
        "f1_weighted": epoch_f1_weighted,
        "kappa": epoch_kappa,
        "cm": cm,
        "report": report,
        "y_true": all_labels,
        "y_pred": all_preds
    }

# -------------------------
# 9) TRAIN STAGE 3C
# -------------------------
best_val_kappa = -1.0
history_stage3c = []

for epoch in range(EPOCHS):
    train_loss, train_acc, train_f1, train_kappa = train_one_epoch_stage3c(
        model, trainloader_stage3c, criterion_stage3c, optimizer_stage3c, device
    )

    val_metrics = evaluate_stage3c(
        model, valloader_stage3c, criterion_stage3c, device, print_report=True
    )

    scheduler_stage3c.step(val_metrics["kappa"])

    history_stage3c.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "train_f1_macro": train_f1,
        "train_kappa": train_kappa,
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["acc"],
        "val_f1_macro": val_metrics["f1_macro"],
        "val_f1_weighted": val_metrics["f1_weighted"],
        "val_kappa": val_metrics["kappa"]
    })

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train Macro F1: {train_f1:.4f} | Train Kappa: {train_kappa:.4f}")
    print(f"Val Loss:   {val_metrics['loss']:.4f} | Val Acc: {val_metrics['acc']:.4f} | Val Macro F1: {val_metrics['f1_macro']:.4f} | Val Weighted F1: {val_metrics['f1_weighted']:.4f} | Val Kappa: {val_metrics['kappa']:.4f}")
    print(f"Current LR: {optimizer_stage3c.param_groups[0]['lr']:.7f}")

    if val_metrics["kappa"] > best_val_kappa:
        best_val_kappa = val_metrics["kappa"]
        torch.save(model.state_dict(), STAGE3C_PATH)
        print(f"Saved best model to {STAGE3C_PATH}")

print("Stage 3C training complete.")

# -------------------------
# 10) TEST BEST STAGE 3C
# -------------------------
print("\nTesting best Stage 3C model...")
model.load_state_dict(torch.load(STAGE3C_PATH, map_location=device))

test_metrics = evaluate_stage3c(
    model, testloader_stage3c, criterion_stage3c, device, print_report=True
)

print("\nTEST METRICS - STAGE 3C")
print("Loss:", test_metrics["loss"])
print("Accuracy:", test_metrics["acc"])
print("Macro F1:", test_metrics["f1_macro"])
print("Weighted F1:", test_metrics["f1_weighted"])
print("Quadratic Kappa:", test_metrics["kappa"])

In [ ]:
# ==========================================
# STAGE-3B + MILD CLASS-AWARE SAMPLING
# Keep the same CB-Focal style loss idea,
# but use a soft sampler to mildly oversample
# classes: Mild (1), Severe (3), Proliferative (4)
# ==========================================

import os
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import models
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score,
    confusion_matrix, classification_report
)

# -----------------------------
# 0) Assumptions from your notebook:
# already available:
# - traindf, valdf, testdf
# - train_dataset, val_dataset, test_dataset
# - valloader, testloader  (or we recreate below)
# - LABELCOL
# - device
# -----------------------------

NUM_CLASSES = 5
CLASS_NAMES = ["NoDR", "Mild", "Moderate", "Severe", "ProliferativeDR"]
BATCH_SIZE = 16
NUM_WORKERS = 2

# -----------------------------------
# 1) Mild class-aware sampler
# -----------------------------------
train_labels = train_df[LABEL_COL].astype(int).values
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)

print("Training class counts:", class_counts)

# Base inverse-frequency weights
base_class_weights = 1.0 / (class_counts + 1e-8)

# Normalize so class 0 starts near 1.0 scale
base_class_weights = base_class_weights / base_class_weights[0]

# Mild extra emphasis only on underperforming minority classes
# Tune gently — avoid huge values
manual_boost = {
    0: 1.00,   # No DR
    1: 1.35,   # Mild
    2: 1.00,   # Moderate
    3: 1.25,   # Severe
    4: 1.35    # Proliferative
}

sampling_class_weights = np.array([
    base_class_weights[c] * manual_boost[c] for c in range(NUM_CLASSES)
], dtype=np.float32)

# Cap overly large weights to avoid unstable oversampling
sampling_class_weights = np.clip(sampling_class_weights, 1.0, 8.0)

print("Sampling class weights:", sampling_class_weights)

sample_weights = sampling_class_weights[train_labels]
sample_weights = torch.DoubleTensor(sample_weights)

train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),   # one epoch ~ same length as training set
    replacement=True
)

trainloader_stage3b_sampler = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    shuffle=False,   # must be False when sampler is used
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# Recreate val/test loaders only if needed
valloader_stage3b = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

testloader_stage3b = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# -----------------------------------
# 2) Same Stage-3B style loss
# -----------------------------------
class ClassBalancedFocalLoss(nn.Module):
    def __init__(self, samples_per_class, num_classes=5, beta=0.999, gamma=1.0):
        super().__init__()
        self.num_classes = num_classes
        self.gamma = gamma

        samples_per_class = np.array(samples_per_class, dtype=np.float32)
        effective_num = 1.0 - np.power(beta, samples_per_class)
        cb_weights = (1.0 - beta) / (effective_num + 1e-8)
        cb_weights = cb_weights / np.sum(cb_weights) * num_classes

        self.class_weights = torch.tensor(cb_weights, dtype=torch.float32)

    def forward(self, logits, targets):
        weights = self.class_weights.to(logits.device)
        ce = F.cross_entropy(logits, targets, reduction='none', weight=weights)
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        return focal.mean()

criterion_stage3b_sampler = ClassBalancedFocalLoss(
    samples_per_class=class_counts,
    num_classes=NUM_CLASSES,
    beta=0.999,
    gamma=1.0
)

# -----------------------------------
# 3) Model: load best previous checkpoint
# Prefer Stage-1 if that was your best generalization
# Else load best Stage-2 if you want to continue from there
# -----------------------------------
model = models.efficientnet_b3(weights=None)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)
model = model.to(device)

candidate_ckpts = [
    "/kaggle/working/best_efficientnet_b3_stage3b_balanced.pth"
]

loaded_ckpt = None
for ckpt in candidate_ckpts:
    if os.path.exists(ckpt):
        model.load_state_dict(torch.load(ckpt, map_location=device))
        loaded_ckpt = ckpt
        break

if loaded_ckpt is None:
    raise FileNotFoundError("No previous checkpoint found for Stage-3B sampler experiment.")

print(f"Loaded checkpoint: {loaded_ckpt}")

# -----------------------------------
# 4) Optimizer + scheduler
# Keep LR small because this is a refinement run
# -----------------------------------
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=1
)

# -----------------------------------
# 5) Train / eval helpers
# -----------------------------------
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    epoch_f1_weighted = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
    epoch_kappa = cohen_kappa_score(all_labels, all_preds, weights="quadratic")

    return {
        "loss": epoch_loss,
        "acc": epoch_acc,
        "f1_macro": epoch_f1_macro,
        "f1_weighted": epoch_f1_weighted,
        "kappa": epoch_kappa
    }

@torch.no_grad()
def evaluate(model, loader, criterion, device, print_report=False):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    epoch_f1_weighted = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
    epoch_kappa = cohen_kappa_score(all_labels, all_preds, weights="quadratic")
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(NUM_CLASSES)))
    report = classification_report(
        all_labels,
        all_preds,
        labels=list(range(NUM_CLASSES)),
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0
    )

    if print_report:
        print("Confusion Matrix (rows=true, cols=pred):")
        print(cm)
        print("\nClassification Report:\n")
        print(report)

    return {
        "loss": epoch_loss,
        "acc": epoch_acc,
        "f1_macro": epoch_f1_macro,
        "f1_weighted": epoch_f1_weighted,
        "kappa": epoch_kappa,
        "cm": cm,
        "report": report,
        "y_true": all_labels,
        "y_pred": all_preds
    }

# -----------------------------------
# 6) Short refinement run
# Main selection metric = validation kappa
# -----------------------------------
EPOCHS = 3
best_val_kappa = -np.inf
best_path = "/kaggle/working/bestefficientnetb3_stage3b_mildsampler.pth"
best_state = None

for epoch in range(EPOCHS):
    train_metrics = train_one_epoch(
        model,
        trainloader_stage3b_sampler,
        criterion_stage3b_sampler,
        optimizer,
        device
    )

    val_metrics = evaluate(
        model,
        valloader_stage3b,
        criterion_stage3b_sampler,
        device,
        print_report=True
    )

    scheduler.step(val_metrics["kappa"])

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(
        f"Train Loss: {train_metrics['loss']:.4f} | "
        f"Train Acc: {train_metrics['acc']:.4f} | "
        f"Train Macro F1: {train_metrics['f1_macro']:.4f} | "
        f"Train Weighted F1: {train_metrics['f1_weighted']:.4f} | "
        f"Train Kappa: {train_metrics['kappa']:.4f}"
    )
    print(
        f"Val Loss:   {val_metrics['loss']:.4f} | "
        f"Val Acc: {val_metrics['acc']:.4f} | "
        f"Val Macro F1: {val_metrics['f1_macro']:.4f} | "
        f"Val Weighted F1: {val_metrics['f1_weighted']:.4f} | "
        f"Val Kappa: {val_metrics['kappa']:.4f}"
    )

    if val_metrics["kappa"] > best_val_kappa:
        best_val_kappa = val_metrics["kappa"]
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, best_path)
        print(f"Saved best model to: {best_path}")

# -----------------------------------
# 7) Final test evaluation
# -----------------------------------
print("\nLoading best mild-sampler model for final test...")
model.load_state_dict(torch.load(best_path, map_location=device))

test_metrics = evaluate(
    model,
    testloader_stage3b,
    criterion_stage3b_sampler,
    device,
    print_report=True
)

print("\nFINAL TEST METRICS — STAGE-3B + MILD CLASS-AWARE SAMPLING")
print(f"Loss:        {test_metrics['loss']:.4f}")
print(f"Accuracy:    {test_metrics['acc']:.4f}")
print(f"Macro F1:    {test_metrics['f1_macro']:.4f}")
print(f"Weighted F1: {test_metrics['f1_weighted']:.4f}")
print(f"Kappa:       {test_metrics['kappa']:.4f}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
import copy
import os
from torchvision import models
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix, classification_report

# ---------------------------------------------------------
# 0. Optional: reduce batch size in your DataLoader cell if needed
#    Example:
#    BATCH_SIZE = 8 or 4
# ---------------------------------------------------------

# ---------------------------------------------------------
# 1. Rebuild EfficientNet-B3 and load best previous checkpoint ONCE
#    Based on your notebook naming, loaders are:
#    trainloader, valloader, testloader
#    dataframe names are:
#    traindf, valdf, testdf
#    label column is:
#    LABELCOL
# ---------------------------------------------------------
NUM_CLASSES = 5
CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative"]

save_dir = "/kaggle/working"
os.makedirs(save_dir, exist_ok=True)

base_model = models.efficientnet_b3(weights=None)
in_features = base_model.classifier[1].in_features
base_model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)
base_model = base_model.to(device)

# Use your existing saved checkpoint as starting point
ckpt_candidates = [
    os.path.join(save_dir, "best_efficientnet_b3_stage3b_balanced.pth"),
    
]

loaded = False
loaded_ckpt = None
for ckpt in ckpt_candidates:
    if os.path.exists(ckpt):
        base_model.load_state_dict(torch.load(ckpt, map_location=device))
        print(f"Loaded checkpoint: {ckpt}")
        loaded = True
        loaded_ckpt = ckpt
        break

if not loaded:
    raise FileNotFoundError("No checkpoint found in /kaggle/working to initialize the model.")

# ---------------------------------------------------------
# 2. Loss definitions
# ---------------------------------------------------------
train_labels = train_df[LABEL_COL].astype(int).values
samples_per_class = np.bincount(train_labels, minlength=NUM_CLASSES)

class ClassBalancedFocalLoss(nn.Module):
    def __init__(self, samples_per_class, num_classes=5, beta=0.999, gamma=1.5):
        super().__init__()
        self.num_classes = num_classes
        self.gamma = gamma

        samples_per_class = np.array(samples_per_class, dtype=np.float32)
        effective_num = 1.0 - np.power(beta, samples_per_class)
        weights = (1.0 - beta) / (effective_num + 1e-8)
        weights = weights / weights.sum() * num_classes
        self.class_weights = torch.tensor(weights, dtype=torch.float32)

    def forward(self, logits, targets):
        weights = self.class_weights.to(logits.device)
        ce = F.cross_entropy(logits, targets, reduction='none', weight=weights)
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        return focal.mean()

class HybridCELoss(nn.Module):
    def __init__(self, samples_per_class, num_classes=5, beta=0.999, gamma=1.5, alpha=0.6):
        super().__init__()
        self.alpha = alpha
        self.cb_focal = ClassBalancedFocalLoss(
            samples_per_class=samples_per_class,
            num_classes=num_classes,
            beta=beta,
            gamma=gamma
        )

        samples_per_class = np.array(samples_per_class, dtype=np.float32)
        effective_num = 1.0 - np.power(beta, samples_per_class)
        weights = (1.0 - beta) / (effective_num + 1e-8)
        weights = weights / weights.sum() * num_classes
        self.ce_weights = torch.tensor(weights, dtype=torch.float32)

    def forward(self, logits, targets):
        weights = self.ce_weights.to(logits.device)
        ce_loss = F.cross_entropy(logits, targets, weight=weights)
        focal_loss = self.cb_focal(logits, targets)
        return self.alpha * ce_loss + (1.0 - self.alpha) * focal_loss

# ---------------------------------------------------------
# 3. Train / eval helpers
# ---------------------------------------------------------
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in dataloader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

        del outputs, loss
        torch.cuda.empty_cache()

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    epoch_kappa = cohen_kappa_score(all_labels, all_preds, weights="quadratic")

    return epoch_loss, epoch_acc, epoch_f1_macro, epoch_kappa


@torch.no_grad()
def evaluate(model, dataloader, criterion, device, print_report=False):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in dataloader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

        del outputs, loss
        torch.cuda.empty_cache()

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    epoch_f1_weighted = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
    epoch_kappa = cohen_kappa_score(all_labels, all_preds, weights="quadratic")

    cm = confusion_matrix(all_labels, all_preds, labels=list(range(NUM_CLASSES)))
    report = classification_report(
        all_labels,
        all_preds,
        labels=list(range(NUM_CLASSES)),
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0
    )

    if print_report:
        print("Confusion Matrix:")
        print(cm)
        print("\nClassification Report:\n")
        print(report)

    return {
        "loss": epoch_loss,
        "acc": epoch_acc,
        "f1_macro": epoch_f1_macro,
        "f1_weighted": epoch_f1_weighted,
        "kappa": epoch_kappa,
        "cm": cm,
        "report": report
    }

# ---------------------------------------------------------
# 4. Run one gamma at a time and SAVE best checkpoint for each gamma
# ---------------------------------------------------------
gamma_values = [1.0, 1.5, 2.0]   # shrink to [1.5] if memory/time is tight
alpha = 0.6
results = []

best_overall_val_kappa = -1
best_overall_path = os.path.join(save_dir, "best_efficientnet_b3_stage4_hybrid_overall.pth")
best_overall_info = None

for gamma in gamma_values:
    model = copy.deepcopy(base_model).to(device)
    torch.cuda.empty_cache()

    print("\n==============================")
    print(f"Hybrid CE + CB-Focal: gamma={gamma}, alpha={alpha}")
    print("==============================")

    criterion = HybridCELoss(
        samples_per_class=samples_per_class,
        num_classes=NUM_CLASSES,
        beta=0.999,
        gamma=gamma,
        alpha=alpha
    )

    optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=1
    )

    best_val_kappa = -1
    best_state = copy.deepcopy(model.state_dict())
    gamma_ckpt_path = os.path.join(save_dir, f"best_efficientnet_b3_stage4_hybrid_gamma_{gamma:.1f}.pth")

    for epoch in range(3):   # reduce to 2 if runtime is too high
        train_loss, train_acc, train_f1, train_kappa = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )

        val_metrics = evaluate(model, val_loader, criterion, device, print_report=False)
        scheduler.step(val_metrics["kappa"])

        print(f"Epoch {epoch+1}/3")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train Macro F1: {train_f1:.4f} | Train Kappa: {train_kappa:.4f}")
        print(f"Val Loss:   {val_metrics['loss']:.4f} | Val Acc: {val_metrics['acc']:.4f} | Val Macro F1: {val_metrics['f1_macro']:.4f} | Val Kappa: {val_metrics['kappa']:.4f}")

        if val_metrics["kappa"] > best_val_kappa:
            best_val_kappa = val_metrics["kappa"]
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, gamma_ckpt_path)
            print(f"Saved best checkpoint for gamma={gamma} at: {gamma_ckpt_path}")

        if val_metrics["kappa"] > best_overall_val_kappa:
            best_overall_val_kappa = val_metrics["kappa"]
            torch.save(model.state_dict(), best_overall_path)
            best_overall_info = {
                "gamma": gamma,
                "alpha": alpha,
                "val_kappa": val_metrics["kappa"],
                "path": best_overall_path
            }
            print(f"Saved NEW overall best model at: {best_overall_path}")

    # load best state for this gamma before test evaluation
    model.load_state_dict(best_state)
    test_metrics = evaluate(model, test_loader, criterion, device, print_report=True)

    results.append({
        "gamma": gamma,
        "alpha": alpha,
        "source_checkpoint": loaded_ckpt,
        "saved_gamma_checkpoint": gamma_ckpt_path,
        "best_val_kappa": best_val_kappa,
        "test_acc": test_metrics["acc"],
        "test_f1_macro": test_metrics["f1_macro"],
        "test_f1_weighted": test_metrics["f1_weighted"],
        "test_kappa": test_metrics["kappa"],
    })

    del model
    torch.cuda.empty_cache()

# ---------------------------------------------------------
# 5. Save results summary CSV
# ---------------------------------------------------------
results_df = pd.DataFrame(results)
results_csv_path = os.path.join(save_dir, "efficientnet_b3_stage4_hybrid_results_summary.csv")
results_df.to_csv(results_csv_path, index=False)

# ---------------------------------------------------------
# 6. Print summary
# ---------------------------------------------------------
print("\n==============================")
print("HYBRID LOSS RESULTS SUMMARY")
print("==============================")
for r in results:
    print(
        f"gamma={r['gamma']:.1f} | alpha={r['alpha']:.1f} | "
        f"best_val_kappa={r['best_val_kappa']:.4f} | "
        f"test_acc={r['test_acc']:.4f} | "
        f"test_macro_f1={r['test_f1_macro']:.4f} | "
        f"test_weighted_f1={r['test_f1_weighted']:.4f} | "
        f"test_kappa={r['test_kappa']:.4f}"
    )

print("\nSaved per-gamma checkpoints and summary:")
for r in results:
    print(f"gamma={r['gamma']:.1f} -> {r['saved_gamma_checkpoint']}")

print(f"\nOverall best checkpoint: {best_overall_path}")
print(f"Results CSV: {results_csv_path}")

if best_overall_info is not None:
    print("\nBest overall model info:")
    print(best_overall_info)

Loaded checkpoint: /kaggle/working/best_efficientnet_b3_stage3b_balanced.pth

Hybrid CE + CB-Focal: gamma=1.0, alpha=0.6
Epoch 1/3
Train Loss: 0.3059 | Train Acc: 0.8755 | Train Macro F1: 0.7231 | Train Kappa: 0.8455
Val Loss:   0.4159 | Val Acc: 0.8549 | Val Macro F1: 0.6061 | Val Kappa: 0.8104
Saved best checkpoint for gamma=1.0 at: /kaggle/working/best_efficientnet_b3_stage4_hybrid_gamma_1.0.pth
Saved NEW overall best model at: /kaggle/working/best_efficientnet_b3_stage4_hybrid_overall.pth
Epoch 2/3
Train Loss: 0.2953 | Train Acc: 0.8784 | Train Macro F1: 0.7318 | Train Kappa: 0.8508
Val Loss:   0.4214 | Val Acc: 0.8528 | Val Macro F1: 0.6079 | Val Kappa: 0.8140
Saved best checkpoint for gamma=1.0 at: /kaggle/working/best_efficientnet_b3_stage4_hybrid_gamma_1.0.pth
Saved NEW overall best model at: /kaggle/working/best_efficientnet_b3_stage4_hybrid_overall.pth
Epoch 3/3
Train Loss: 0.2880 | Train Acc: 0.8817 | Train Macro F1: 0.7442 | Train Kappa: 0.8557
Val Loss:   0.4326 | Val Acc:

KeyboardInterrupt: 